# 🔬 KoChatGPT 업그레이드 — Ablation Study (변인 통제 실험)

과학실험처럼 **한 번에 변수 하나만 바꿔서** 각 개선 전략의 기여도를 분리 측정합니다.

## 실험 설계

**통제 변인(모든 실험 공통·고정)**: seed=42, 동일한 held-out 테스트셋 60개, 동일한 문자 단위 BLEU/ROUGE 메트릭

| 실험 | 조작 변인 | 가설 | T4 예상 소요 |
|---|---|---|---|
| **E0** (대조군) | — | baseline | ~15분 |
| **E1** | 데이터 정제 **OFF** (원본 그대로) | 정제가 성능을 올렸다 | ~15분 |
| **E4** | 디코딩을 **greedy로 고정** | 샘플링 디코딩의 기여분 | ~3분 (E0 모델 재사용) |
| **E5** | 모델을 **ko-gpt-trinity-1.2B(8bit)** 로 교체 | 지식 부족의 원인은 모델 크기 | ~40-60분 |

## 사용법

1. 바로 아래 셀의 `EXPERIMENT` 값만 바꾸고 → **런타임 전체 재실행** (Ctrl+F9)
2. 결과는 `results/experiments.csv`에 **자동 누적**되고, 마지막 셀이 지금까지의 모든 실험을 비교표 + 그래프로 그려줍니다
3. 추천 실행 순서: **같은 런타임에서 E0 → E4** (E4는 E0의 어댑터를 재사용해서 학습 생략) → E1 → E5는 메모리를 많이 쓰므로 **새 런타임**에서
4. ⚠️ Colab 런타임이 초기화되면 로컬 파일이 사라집니다. 실험 사이에 런타임을 껐다면 `results/` 폴더와 `adapters/` 폴더를 다운로드해뒀다가 다시 업로드하세요 (마지막 셀 참고)

> ⚙️ 런타임 → 런타임 유형 변경 → **T4 GPU**


## 0. 실험 스위치 — 여기만 바꾸세요

In [ ]:
# ═══════════════════════════════════════════════════
EXPERIMENT = 'E0'      # ← 'E0' | 'E1' | 'E4' | 'E5'
# ═══════════════════════════════════════════════════

EXP_CONFIGS = {
    'E0': dict(desc='대조군: 정제 데이터 + KoGPT-2 + LoRA + 샘플링 디코딩',
               model='skt/kogpt2-base-v2', clean=True,  decode='sampling',
               use_8bit=False, max_train=3000, batch=16, accum=2, epochs=3),
    'E1': dict(desc='정제 OFF: 원본 데이터 그대로 학습 (나머지는 E0과 동일)',
               model='skt/kogpt2-base-v2', clean=False, decode='sampling',
               use_8bit=False, max_train=3000, batch=16, accum=2, epochs=3),
    'E4': dict(desc='디코딩 ablation: greedy 고정 (E0 어댑터 재사용)',
               model='skt/kogpt2-base-v2', clean=True,  decode='greedy',
               use_8bit=False, max_train=3000, batch=16, accum=2, epochs=3),
    'E5': dict(desc='모델 교체: ko-gpt-trinity-1.2B + 8bit 양자화 + LoRA',
               model='skt/ko-gpt-trinity-1.2B-v0.5', clean=True, decode='sampling',
               use_8bit=True, max_train=1500, batch=4, accum=8, epochs=3),
}
CFG = EXP_CONFIGS[EXPERIMENT]

# 디코딩 설정 (E0의 디코딩 서치에서 다른 조합이 이겼다면 'sampling'을 그 값으로 바꿔주세요)
DECODE_KW = {
    'sampling': dict(do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.2),
    'greedy'  : dict(do_sample=False, repetition_penalty=1.2),
}[CFG['decode']]

# 학습된 LoRA 어댑터 저장 키: (모델, 정제여부)가 같으면 재사용 → E4가 E0 학습을 생략하는 원리
MODEL_TAG   = 'trinity' if 'trinity' in CFG['model'] else 'kogpt2'
ADAPTER_KEY = f"{MODEL_TAG}_{'clean' if CFG['clean'] else 'raw'}"

print(f"▶ 실험 {EXPERIMENT}: {CFG['desc']}")
print(f"  adapter key: {ADAPTER_KEY} / decode: {CFG['decode']}")

## 1. 환경 설정

In [ ]:
import time
RUN_START = time.time()

!pip install -q transformers==4.46.3 peft==0.15.2 accelerate==1.1.1
if CFG['use_8bit']:
    !pip install -q bitsandbytes

import torch, random, os, re, json as pyjson
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

assert torch.cuda.is_available(), "⚠️ 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 다시 실행하세요."
os.makedirs('results', exist_ok=True); os.makedirs('adapters', exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0))

## 2. 데이터 준비

핵심 통제 장치: **테스트셋 60개는 어떤 실험이든 항상 "정제 파이프라인 + seed 42" 기준으로 동일하게** 뽑습니다.
E1(정제 OFF)은 *학습 데이터만* 원본을 쓰되, 테스트셋과 겹치는 prompt는 제외해 leakage를 막습니다.

In [ ]:
if not os.path.exists('KoChatGPT'):
    !git clone -q https://github.com/airobotlab/KoChatGPT.git

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        txt = f.read().strip()
    try:
        data = pyjson.loads(txt)
        return data if isinstance(data, list) else [data]
    except pyjson.JSONDecodeError:
        return [pyjson.loads(l) for l in txt.splitlines() if l.strip()]

sft_raw = load_jsonl('KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl')

# ── 정제 파이프라인 (E0과 동일 + 답변 맨 앞 따옴표 아티팩트 제거 규칙 추가) ──
refusal_pat = re.compile(r'(AI\s?언어\s?모델|인공지능 (챗봇|모델|어시스턴트))')

def clean_text(t):
    t = re.sub(r'https?://\S+', '', t)
    t = re.sub(r'(.)\1{4,}', r'\1\1\1', t)
    t = re.sub(r'\s+', ' ', t).strip()
    t = re.sub(r'^[\'\"`]+', '', t)      # ← 답변 앞 스트레이 따옴표 제거 (E0 데모에서 발견된 아티팩트)
    return t.strip()

def is_bad(p, c):
    if len(p) < 2 or len(c) < 10 or len(c) > 400: return True
    if refusal_pat.search(c) and '할 수 없' in c and len(c) < 120: return True
    return False

seen, cleaned = set(), []
for d in sft_raw:
    p, c = clean_text(d['prompt']), clean_text(d['completion'])
    if (p, c) in seen or is_bad(p, c): continue
    seen.add((p, c)); cleaned.append({'prompt': p, 'completion': c})

AUG_TEMPLATES = [lambda p: f"다음 질문에 답해줘. {p}",
                 lambda p: f"{p} 알기 쉽게 설명해줘.",
                 lambda p: f"질문: {p}"]
random.shuffle(cleaned)
n_aug = int(len(cleaned) * 0.2)
augmented = [{'prompt': random.choice(AUG_TEMPLATES)(d['prompt']), 'completion': d['completion']}
             for d in random.sample(cleaned, n_aug)]
dataset_all = cleaned + augmented
random.shuffle(dataset_all)

# ── 고정 테스트셋 (모든 실험 공통!) ──
EVAL_N   = 60
test_set = dataset_all[:EVAL_N]
test_prompts = [d['prompt'] for d in test_set]
test_refs    = [d['completion'] for d in test_set]
test_prompt_set = set(test_prompts)

# ── 학습 데이터: 실험에 따라 분기 ──
if CFG['clean']:
    train_set = dataset_all[EVAL_N : EVAL_N + CFG['max_train']]
else:
    raw_pool = [{'prompt': d['prompt'].strip(), 'completion': d['completion'].strip()}
                for d in sft_raw
                if d['prompt'].strip() and d['completion'].strip()
                and clean_text(d['prompt']) not in test_prompt_set]   # leakage 방지만 적용
    random.shuffle(raw_pool)
    train_set = raw_pool[:CFG['max_train']]

print(f"train {len(train_set)} ({'정제' if CFG['clean'] else '원본'}) / test {len(test_set)} (항상 동일)")

## 3. 모델 로드 + LoRA 부착

- KoGPT-2와 trinity 모두 GPT-2 아키텍처라 `target_modules=['c_attn']` 그대로 사용합니다
- E5는 1.2B 모델을 **8bit 양자화**로 로드해 T4 16GB 안에 넣습니다
- `adapters/{key}` 폴더에 학습된 어댑터가 이미 있으면 학습을 **건너뛰고 재사용**합니다 (강제 재학습하려면 해당 폴더 삭제)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

BOS = '<s>' if MODEL_TAG == 'trinity' else '</s>'
tok_kwargs = dict(bos_token=BOS, eos_token='</s>', unk_token='<unk>',
                  pad_token='<pad>', mask_token='<mask>')
try:
    tokenizer = AutoTokenizer.from_pretrained(CFG['model'], **tok_kwargs)
except Exception:
    tokenizer = PreTrainedTokenizerFast.from_pretrained(CFG['model'], **tok_kwargs)

if CFG['use_8bit']:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    base_model = AutoModelForCausalLM.from_pretrained(
        CFG['model'],
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map='auto')
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(CFG['model']).cuda()

lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32,
                      lora_dropout=0.05, target_modules=['c_attn'], fan_in_fan_out=True)

ADAPTER_DIR = f'adapters/{ADAPTER_KEY}'
NEED_TRAIN  = not os.path.exists(ADAPTER_DIR)
if NEED_TRAIN:
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()
else:
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    print(f"♻️ 기존 어댑터 재사용: {ADAPTER_DIR} (학습 생략)")

## 4. SFT 학습 (필요할 때만)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

MAX_LEN = 256
PROMPT_TMPL = "### 질문: {q}\n\n### 답변:"

class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.examples = [tokenizer(PROMPT_TMPL.format(q=d['prompt']) + ' ' + d['completion'] + tokenizer.eos_token,
                                   truncation=True, max_length=MAX_LEN)['input_ids']
                         for d in data]
    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return {'input_ids': self.examples[i]}

train_min = 0.0
if NEED_TRAIN:
    train_ds = SFTDataset(train_set)
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    args = TrainingArguments(
        output_dir='tmp_out',
        num_train_epochs=CFG['epochs'],
        per_device_train_batch_size=CFG['batch'],
        gradient_accumulation_steps=CFG['accum'],
        learning_rate=2e-4, warmup_ratio=0.05,
        fp16=True, logging_steps=50,
        gradient_checkpointing=CFG['use_8bit'],
        save_strategy='no', report_to='none', seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
    t0 = time.time()
    trainer.train()
    train_min = (time.time() - t0) / 60
    model.save_pretrained(ADAPTER_DIR)
    print(f"✅ 학습 완료 {train_min:.1f}분 → {ADAPTER_DIR} 저장")
else:
    print("학습 생략 (어댑터 재사용)")

## 5. 메트릭 & 생성 함수 (모든 실험 공통·고정)

In [ ]:
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smooth = SmoothingFunction().method1

def char_tokens(t): return list(t.replace(' ', ''))
def ngrams(toks, n): return Counter(tuple(toks[i:i+n]) for i in range(len(toks)-n+1))

def rouge_n(ref, hyp, n):
    r, h = ngrams(char_tokens(ref), n), ngrams(char_tokens(hyp), n)
    if not r or not h: return 0.0
    ov = sum((r & h).values())
    p, rec = ov / max(sum(h.values()), 1), ov / max(sum(r.values()), 1)
    return 2*p*rec/(p+rec) if p+rec else 0.0

def rouge_l(ref, hyp):
    a, b = char_tokens(ref), char_tokens(hyp)
    if not a or not b: return 0.0
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[-1][-1]; p, rec = lcs/len(b), lcs/len(a)
    return 2*p*rec/(p+rec) if p+rec else 0.0

def bleu(ref, hyp):
    r, h = char_tokens(ref), char_tokens(hyp)
    return sentence_bleu([r], h, weights=(0.5, 0.5), smoothing_function=smooth) if h else 0.0

def distinct_n(texts, n=2):
    allg = [g for t in texts for g in
            [tuple(char_tokens(t)[i:i+n]) for i in range(len(char_tokens(t))-n+1)]]
    return len(set(allg)) / max(len(allg), 1)

def score_set(refs, hyps):
    return {'BLEU'   : float(np.mean([bleu(r, h)       for r, h in zip(refs, hyps)])),
            'ROUGE-1': float(np.mean([rouge_n(r, h, 1) for r, h in zip(refs, hyps)])),
            'ROUGE-2': float(np.mean([rouge_n(r, h, 2) for r, h in zip(refs, hyps)])),
            'ROUGE-L': float(np.mean([rouge_l(r, h)    for r, h in zip(refs, hyps)])),
            'Distinct-2': float(distinct_n(hyps))}

@torch.no_grad()
def generate_batch(prompts, use_adapter=True, batch_size=8, max_new_tokens=64, **gen_kwargs):
    model.eval()
    tokenizer.padding_side = 'left'
    if CFG['use_8bit']: batch_size = 4
    outs = []
    for i in range(0, len(prompts), batch_size):
        batch = [PROMPT_TMPL.format(q=p) for p in prompts[i:i+batch_size]]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LEN).to(model.device)
        def _gen():
            return model.generate(**enc, max_new_tokens=max_new_tokens,
                                  eos_token_id=tokenizer.eos_token_id,
                                  pad_token_id=tokenizer.pad_token_id, **gen_kwargs)
        g = _gen() if use_adapter else None
        if g is None:
            with model.disable_adapter():
                g = _gen()
        outs += [tokenizer.decode(g[j][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
                 for j in range(len(batch))]
    tokenizer.padding_side = 'right'
    return outs

print('준비 완료')

## 6. 평가 → `results/experiments.csv` 에 누적 기록

같은 실험을 다시 돌리면 해당 행을 덮어씁니다.

In [ ]:
import pandas as pd

t0 = time.time()
hyps_sft = generate_batch(test_prompts, use_adapter=True, **DECODE_KW)
print(f"SFT 생성 완료 ({(time.time()-t0)/60:.1f}분)")

scores = score_set(test_refs, hyps_sft)
row = {'exp': EXPERIMENT, 'desc': CFG['desc'], 'model': MODEL_TAG,
       'clean': CFG['clean'], 'decode': CFG['decode'],
       **{k: round(v, 4) for k, v in scores.items()},
       'train_min': round(train_min, 1)}

# E0일 때만 Base(파인튜닝 전) 기준선도 함께 기록
rows = [row]
if EXPERIMENT == 'E0':
    hyps_base = generate_batch(test_prompts, use_adapter=False, **DECODE_KW)
    rows.append({'exp': 'Base', 'desc': '파인튜닝 전 KoGPT-2', 'model': MODEL_TAG,
                 'clean': '-', 'decode': CFG['decode'],
                 **{k: round(v, 4) for k, v in score_set(test_refs, hyps_base).items()},
                 'train_min': 0.0})

CSV = 'results/experiments.csv'
df = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame()
for r in rows:
    df = df[df['exp'] != r['exp']] if len(df) else df   # 같은 실험 재실행 시 덮어쓰기
    df = pd.concat([df, pd.DataFrame([r])], ignore_index=True)
df.to_csv(CSV, index=False)

# 정성 비교용 생성문도 저장
with open(f'results/gen_{EXPERIMENT}.json', 'w', encoding='utf-8') as f:
    pyjson.dump({'prompts': test_prompts[:10], 'refs': test_refs[:10], 'hyps': hyps_sft[:10]},
                f, ensure_ascii=False, indent=1)

print(df[['exp', 'BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Distinct-2', 'train_min']].to_string(index=False))

## 7. 데모 질문 4개 — 정성 평가 기록

In [ ]:
demo_questions = [
    "불고기용 고기 한우에요?",
    "리처드 닉슨이 43대 부통령직을 수행한 년도는?",
    "시카고 오헤어 국제공항은 어디에 있어?",
    "오늘 미세먼지 어때?",
]
demo_out = generate_batch(demo_questions, use_adapter=True, max_new_tokens=80, **DECODE_KW)

with open(f'results/demo_{EXPERIMENT}.json', 'w', encoding='utf-8') as f:
    pyjson.dump(dict(zip(demo_questions, demo_out)), f, ensure_ascii=False, indent=1)

for q, a in zip(demo_questions, demo_out):
    print('='*70)
    print(f"❓ {q}")
    print(f"  [{EXPERIMENT}] {a}")

print(f"\n⏱️ 이번 실험 총 소요: {(time.time()-RUN_START)/60:.1f}분")

## 8. 지금까지의 모든 실험 종합 비교

실험을 하나 마칠 때마다 이 셀이 누적된 결과를 그려줍니다.

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv('results/experiments.csv')
order = [e for e in ['Base', 'E0', 'E1', 'E4', 'E5'] if e in df['exp'].values]
df = df.set_index('exp').loc[order].reset_index()

print(df[['exp', 'desc', 'BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Distinct-2', 'train_min']].to_string(index=False))

metrics = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
x = np.arange(len(metrics)); w = 0.8 / max(len(df), 1)
colors = {'Base':'#999999', 'E0':'#4C72B0', 'E1':'#DD8452', 'E4':'#55A868', 'E5':'#C44E52'}
plt.figure(figsize=(9, 4))
for i, (_, r) in enumerate(df.iterrows()):
    plt.bar(x + i*w, [r[m] for m in metrics], w, label=r['exp'],
            color=colors.get(r['exp'], '#8172B3'))
plt.xticks(x + w*(len(df)-1)/2, metrics)
plt.legend(); plt.title('Ablation: metric comparison (char-level, fixed test set)')
plt.tight_layout(); plt.show()

# 데모 질문 side-by-side
demo_files = sorted([f for f in os.listdir('results') if f.startswith('demo_')])
if demo_files:
    demos = {f[5:-5]: pyjson.load(open(f'results/{f}', encoding='utf-8')) for f in demo_files}
    qs = list(next(iter(demos.values())).keys())
    for q in qs:
        print('='*70); print(f"❓ {q}")
        for exp in order:
            if exp in demos:
                print(f"  [{exp:>3}] {demos[exp].get(q, '-')}")

In [ ]:
# (선택) 런타임 종료 전 결과 백업 — BACKUP=True로 바꾸고 실행하면 zip 다운로드
BACKUP = False
if BACKUP:
    !zip -qr ablation_backup.zip results adapters
    from google.colab import files
    files.download('ablation_backup.zip')
# 새 런타임에서 이어서 할 때: zip을 업로드 후 → !unzip -qo ablation_backup.zip

## 9. 결과 해석 가이드 (보고서 작성용)

| 비교 | 읽는 법 |
|---|---|
| **E0 vs Base** | SFT+LoRA 자체의 효과 (이미 확인: ROUGE-1 기준 2.5배) |
| **E0 vs E1** | E0 > E1 이면 "데이터 정제가 성능 향상에 기여했다"는 직접 증거. 특히 E1의 데모 답변에 "저는 AI 언어모델입니다" 류의 회피 응답이 더 자주 나오는지 확인해보세요 |
| **E0 vs E4** | 샘플링 디코딩 vs greedy. Distinct-2 차이가 크고 ROUGE 차이가 작다면 "디코딩은 다양성에 주로 기여"라고 해석할 수 있습니다 |
| **E0 vs E5** | 모델 크기(125M → 1.2B)의 효과. 정량 지표보다 **데모 질문의 사실성**(닉슨, 오헤어 공항)이 개선되는지가 핵심 관전 포인트입니다 |

**보고서 팁**: "가설 → 실험 설계(통제/조작 변인) → 정량 결과 → 정성 사례 → 결론" 구조로 실험당 한 단락씩 쓰면 완성도 높은 ablation study 보고서가 됩니다.
